# Early prediction on the full event sets

10,000 IEEE 9-bus and 7,999 New England events (PowerFactory RMS simulations), streamed shard by shard. Windows are measured from fault inception. Every model is evaluated on the same 5-fold split, repeated 3 times; intervals are corrected resampled-t 95 % intervals.

Set `TRANPY_SHARDS_DIR` to a folder with one sub-folder of shards per system, or leave it unset to stream the published release.

In [ ]:
import logging, os, warnings
import numpy as np
import pandas as pd
import tranpy
from tranpy.early_prediction import evaluate_window_tables, extract_window_tables, find_knee_point
from tranpy.plotting import plot_early_prediction
from tranpy_core.models import make_pipeline
from pathlib import Path
import matplotlib.pyplot as plt
from tranpy_core.plotting import publication_style, save_figure

FIGURES = Path("outputs")
plt.rcParams["figure.dpi"] = 150

warnings.filterwarnings("ignore", category=FutureWarning)
SHARDS = os.environ.get("TRANPY_SHARDS_DIR")
CACHE = Path.home() / "tranpy-data" / "cache" / "tables"
WINDOWS = (0.05, 0.1, 0.2, 0.4, 0.8)
SYSTEMS = ("ieee_9bus", "new_england")
FIGURES.mkdir(exist_ok=True)
logging.basicConfig(filename=FIGURES / "08_progress.log", level=logging.INFO, format="%(asctime)s %(message)s", force=True)

## Data

In [ ]:
sources = {s: tranpy.open_event_shards(s, source=Path(SHARDS) / s if SHARDS else None) for s in SYSTEMS}
pd.DataFrame({s: {"events": m.n_samples, "shards": len(m.shards), **m.label_counts()} for s, m in sources.items()})

In [ ]:
tables = {s: extract_window_tables(m, reference="fault_start", windows_s=WINDOWS, sequences=True, cache_directory=CACHE)
          for s, m in sources.items()}
{s: {f"{w} s {k}": t.features.shape for (w, k), t in ts.items()} for s, ts in tables.items()}

## Models

In [ ]:
feature_models = {
    "logistic regression": make_pipeline("logistic_regression", max_iter=5000),
    "random forest": make_pipeline("random_forest", seed=0),
    "xgboost": make_pipeline("xgboost", seed=0),
    "lightgbm": make_pipeline("lightgbm", seed=0),
}
network = dict(epochs=150, early_stopping_patience=10, seed=0)
sequence_models = {
    "dense network": make_pipeline("dense_network", hidden_units=(64, 32), **network),
    "lstm": make_pipeline("lstm_network", lstm_units=(32,), dense_units=16, **network),
    "tcn": make_pipeline("tcn_network", n_filters=32, dilation_rates=(1, 2, 4), dense_units=16, **network),
    "transformer": make_pipeline("transformer_network", d_model=32, n_heads=4, n_layers=1, ff_dim=64, dense_units=16, **network),
}

## Run

Every finished (window, model) result is appended to `outputs/08_<system>.csv` and skipped when the notebook runs again, so an interrupted run resumes; progress is logged to `outputs/08_progress.log`.

In [ ]:
results = {system: evaluate_window_tables(tables[system], feature_models, sequence_models=sequence_models,
                                          n_splits=5, n_repeats=3, seed=42,
                                          checkpoint=FIGURES / f"08_{system}.csv")
           for system in SYSTEMS}

In [ ]:
for system, frame in results.items():
    print(system)
    display(frame.pivot_table(index=["input", "model"], columns="window_s", values="accuracy_mean").round(4))

In [ ]:
summary = pd.concat({s: f.set_index(["window_s", "model"])[["accuracy_mean", "accuracy_ci95", "roc_auc_mean", "f1_mean", "fit_time_s"]]
                     for s, f in results.items()})
summary.round(4)

In [ ]:
pd.DataFrame({s: {m: find_knee_point(f, m) for m in f.model.unique()} for s, f in results.items()})

## Accuracy versus observation window

In [ ]:
with publication_style():
    fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.4), sharex=True, sharey="row", gridspec_kw={"wspace": 0.45})
    for row, (system, label) in enumerate([("ieee_9bus", "9-bus, 10,000 events"), ("new_england", "39-bus, 7,999 events")]):
        frame = results[system]
        for col, kind in enumerate(("features", "sequence")):
            plot_early_prediction(frame[frame.input == kind], band="ci95", ax=axes[row, col],
                                  title=f"{label}: {kind} models")
            if col:
                axes[row, col].set_ylabel("")
    save_figure(fig, FIGURES / "08_early_prediction_full.pdf")
plt.show()

## Clearing-based windows at full scale

Same folds, windows measured from the detected clearing (non-causal, see docs/validation.md D9).

In [ ]:
comparison = []
fast = {k: feature_models[k] for k in ("random forest", "lightgbm")}
for system, manifest in sources.items():
    clear = extract_window_tables(manifest, reference="fault_clear", windows_s=WINDOWS, cache_directory=CACHE)
    for reference, ts in (("fault_start", tables[system]), ("fault_clear", clear)):
        frame = evaluate_window_tables({k: v for k, v in ts.items() if k[1] == "features"}, fast, n_splits=5, n_repeats=3, seed=42,
                                       checkpoint=FIGURES / f"08_{system}_{reference}_comparison.csv")
        comparison.append(frame.assign(system=system, reference=reference))
comparison = pd.concat(comparison)
comparison.pivot_table(index=["system", "model", "window_s"], columns="reference", values="accuracy_mean").round(4)